# 06 Forecast Report Workflow

A forecasting report should do more than print model output. It should explain the data pattern, justify the model family, show the fitted model behavior, report accuracy measures, and present forecasts in a table that a decision maker can use.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check_between, check_close, check_columns
from smoothing_utils import (
    accuracy_measures, initial_level_mean, initial_line,
    simple_es, optimize_simple_es,
    holt_trend, optimize_holt, holt_forecast_table,
    holt_winters, optimize_holt_winters, holt_winters_forecast,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')

## Transfer Checklist

1. Plot the time series in time order.
2. Identify level, trend, and seasonality.
3. Decide whether seasonal variation looks additive or multiplicative.
4. Choose a smoothing model family.
5. State how initial values are chosen.
6. Fit or optimize smoothing constants.
7. Report SSE, MAD, MSE, and MAPE.
8. Plot observed values and one-step fitted values.
9. Forecast the requested horizon and report intervals when the course formula supports them.
10. Write a short interpretation that connects the numbers to the problem.

In [ ]:
service = pd.DataFrame({
    'Month': np.arange(1, 25),
    'Orders': [118, 121, 125, 128, 130, 134, 137, 141, 144, 146, 151, 154,
               157, 160, 164, 169, 171, 175, 179, 183, 186, 190, 194, 198],
})
y = service['Orders'].to_numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(service['Month'], y, marker='o')
ax.set_xlabel('Month')
ax.set_ylabel('Replacement part orders')
ax.set_title('Teaching example: replacement part orders')
plt.tight_layout()


The replacement-part order series has an upward trend and no obvious repeated seasonal pattern. This makes Holt's trend-corrected model a reasonable course-model choice.


In [ ]:
l0, b0 = initial_line(y, n=12)
alpha, gamma, fit, metrics = optimize_holt(y, l0=l0, b0=b0)
forecast = holt_forecast_table(fit, alpha=alpha, gamma=gamma, h=3)

print(f'Initial l0={l0:.3f}, b0={b0:.3f}')
print(f'Optimized alpha={alpha:.4f}, gamma={gamma:.4f}')
print('Accuracy measures')
display(pd.Series(metrics).to_frame('value'))
print('Forecast table')
display(forecast.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(service['Month'], y, marker='o', label='Observed')
ax.plot(service['Month'], fit['yhat_one_step'], marker='o', label='One-step Holt forecast')
future_time = np.arange(len(y) + 1, len(y) + 4)
ax.plot(future_time, forecast['forecast'], marker='o', label='Forecast')
ax.fill_between(future_time, forecast['lower_95_PI'], forecast['upper_95_PI'], alpha=0.2, label='Approx. 95% PI')
ax.set_xlabel('Month')
ax.set_ylabel('Replacement part orders')
ax.legend()
plt.tight_layout()


## Report Skeleton

A concise report could say: the series shows an increasing trend, so Holt's trend-corrected exponential smoothing is appropriate. The initial level and trend were estimated by fitting a line to the first year of observations. The smoothing constants were selected to minimize SSE subject to `0 <= alpha <= 1` and `0 <= gamma <= 1`. Forecasts are increasing because the final trend estimate is positive, and prediction intervals quantify uncertainty for individual future monthly demand.


## Professional Package Handoff

For homework questions that ask for the course prediction interval table, keep the course calculation because it exposes the recursion and interval formula. In applied work, you would usually fit the point forecasting model with a package such as `statsmodels`, then make a separate and explicit choice about interval construction.

The code below uses the same initial level and trend as the course calculation so the point forecasts are directly comparable.


In [ ]:
from statsmodels.tsa.holtwinters import Holt

statsmodels_holt = Holt(
    y,
    initialization_method='known',
    initial_level=l0,
    initial_trend=b0,
).fit(optimized=True)

pd.DataFrame({
    'tau': forecast['tau'],
    'course_point_forecast': forecast['forecast'],
    'statsmodels_point_forecast': statsmodels_holt.forecast(3),
}).round(2)

If these point forecasts differ from another software package, first check the initialization method, optimized constants, and whether the software is reporting one-step fitted values or multi-step forecasts. Small differences are often implementation choices, not conceptual disagreements.
